In [8]:
import numpy as np
from model_ranking.dataclass import (
    EPFLTargetConfig
)
from model_ranking.feature_ranking import (
    FeatureBasedTransferRanking,
    compute_class_frequencies,
    get_sampling_indices,
    sample_from_image,
)

In [9]:
# Create an instance of EPFLTargetConfig first
epfl_config = EPFLTargetConfig()
print("Loader type:", type(epfl_config.loader))
print("Loader dataset:", epfl_config.loader.dataset)
epfl_config.loader

Loader type: <class 'model_ranking.dataclass.Pytorch3DUnetLoaderMetaConfig'>
Loader dataset: StandardHDF5Dataset


Pytorch3DUnetLoaderMetaConfig(dataset='StandardHDF5Dataset', batch_size=32, num_workers=8, raw_internal_path='raw', label_internal_path='labels', global_normalization=True, global_percentiles=None, file_paths=('/EPFL/test.h5',), slice_builder=Pytorch3DUnetSliceBuilderConfig(name='SliceBuilder', patch_shape=(1, 256, 256), stride_shape=(1, 256, 256), halo_shape=(0, 32, 32)), transformer={'raw': [{'name': 'Normalize'}, {'name': 'ToTensor', 'expand_dims': True}]}, roi=None)

In [10]:
dataset = EPFLTargetConfig().train_loader.create_config(
    output_dir=None, 
    data_base_path="/scratch/talks/data", 
    phase="train"
)

In [11]:
datasets = FeatureBasedTransferRanking().get_datasets(config=dataset)

TypeError: FeatureBasedTransferRanking.__init__() missing 1 required positional argument: 'config'

In [5]:
class_counts = compute_class_frequencies(datasets[0])

In [6]:
class_counts

{0: tensor(61102516), 1: tensor(3778124)}

In [7]:
indices = []
for i, (_, label) in enumerate(datasets[0]):
    label = np.reshape(label.numpy(), [-1])
    print(f"Label shape for sample {i}: {label.shape}")
    indices.append(get_sampling_indices(sampling_seed=42, class_counts=class_counts, labels=label))
    if i > 3:
        break

Label shape for sample 0: (65536,)
Label shape for sample 1: (65536,)
Label shape for sample 1: (65536,)
Label shape for sample 2: (65536,)
Label shape for sample 2: (65536,)
Label shape for sample 3: (65536,)
Label shape for sample 3: (65536,)
Label shape for sample 4: (65536,)
Label shape for sample 4: (65536,)


In [8]:
indices2 = []
for i, (_, label) in enumerate(datasets[0]):
    label = np.reshape(label.numpy(), [-1])
    print(f"Label shape for sample {i}: {label.shape}")
    indices2.append(get_sampling_indices(sampling_seed=42, class_counts=class_counts, labels=label))
    if i > 3:
        break

Label shape for sample 0: (65536,)
Label shape for sample 1: (65536,)
Label shape for sample 1: (65536,)
Label shape for sample 2: (65536,)
Label shape for sample 2: (65536,)
Label shape for sample 3: (65536,)
Label shape for sample 3: (65536,)
Label shape for sample 4: (65536,)
Label shape for sample 4: (65536,)


In [9]:
for i in range(len(indices)):
    print(np.all(indices[i] == indices2[i]))  # Check if the sampling indices are consistent across samples

True
True
True
True
True


In [1]:
import torch

from CCFV.utils.sliding_window_sampling import FeatureExtractor
from pytorch3dunet.unet3d.model import UNet2D
from model_ranking.dataclass import (
    ModelSourceConfig,
)

In [2]:
model_config = {
    "source_name": "EPFL",
    "model_name": "E_model5",
    "model_type": "UNet2D"
}
model_cfg = ModelSourceConfig.model_validate(model_config)
model_cfg = model_cfg.create_config(feature_perturbation=None)

In [3]:
model_cfg = model_cfg.model_copy(update={"feature_return": True})

In [30]:
model_cfg.feature_return

True

In [31]:
model = UNet2D(**model_cfg.model_dump())
model = model.eval()
print(model)

UNet2D(
  (encoders): ModuleList(
    (0): Encoder(
      (basic_module): DoubleConv(
        (SingleConv1): SingleConv(
          (batchnorm): BatchNorm2d(1, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (ReLU): ReLU(inplace=True)
        )
        (SingleConv2): SingleConv(
          (batchnorm): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (ReLU): ReLU(inplace=True)
        )
      )
    )
    (1): Encoder(
      (pooling): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (basic_module): DoubleConv(
        (SingleConv1): SingleConv(
          (batchnorm): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv): Conv2d(32, 32, kernel_size=(3, 3), strid

In [37]:
layers = [
    'decoders.2.basic_module.SingleConv2.ReLU',
]
feature_extractor = FeatureExtractor(model, layers)

In [38]:
with torch.no_grad():
    for i, (image, label) in enumerate(datasets[0]):
        print(image.shape, label.shape)
        #image = torch.squeeze(image, dim=-3)  # Ensure image is 4D
        features = feature_extractor(image)
        print("extracted_features", features[layers[0]].shape)

        pred, feats = model(image)
        print("output_features", feats[-1].shape)
        print("Features equal:", torch.allclose(features[layers[0]], feats[-1].detach(), atol=1e-5))

        sampled_output, sampled_labels = sample_from_image(
            features[layers[0]].detach().numpy(),
            label.numpy(), 
            42, 
            class_counts, 
            1000
        )
        print("Sampled output shape:", sampled_output.shape)
        print("Sampled labels shape:", sampled_labels.shape)
        if i > 3:
            break
    feature_extractor.remove_handler()

torch.Size([1, 1, 256, 256]) torch.Size([1, 1, 256, 256])
extracted_features torch.Size([1, 32, 256, 256])
output_features torch.Size([1, 32, 256, 256])
Features equal: True
Sampled output shape: (1000, 32)
Sampled labels shape: (1000,)
torch.Size([1, 1, 256, 256]) torch.Size([1, 1, 256, 256])
extracted_features torch.Size([1, 32, 256, 256])
output_features torch.Size([1, 32, 256, 256])
Features equal: True
Sampled output shape: (1000, 32)
Sampled labels shape: (1000,)
torch.Size([1, 1, 256, 256]) torch.Size([1, 1, 256, 256])
extracted_features torch.Size([1, 32, 256, 256])
output_features torch.Size([1, 32, 256, 256])
Features equal: True
Sampled output shape: (1000, 32)
Sampled labels shape: (1000,)
torch.Size([1, 1, 256, 256]) torch.Size([1, 1, 256, 256])
extracted_features torch.Size([1, 32, 256, 256])
output_features torch.Size([1, 32, 256, 256])
Features equal: True
Sampled output shape: (1000, 32)
Sampled labels shape: (1000,)
torch.Size([1, 1, 256, 256]) torch.Size([1, 1, 256, 

In [19]:
labels, counts = np.unique(sampled_labels, return_counts=True)
print(dict(zip(labels, counts)))

{0.0: 492, 1.0: 508}


In [35]:
# Let's test the difference between decoders.2 and decoders.2.basic_module.SingleConv2.ReLU
layers_test = [
    'decoders.2',
    'decoders.2.basic_module.SingleConv2.ReLU'
]

feature_extractor_test = FeatureExtractor(model, layers_test)

with torch.no_grad():
    for i, (image, label) in enumerate(datasets[0]):
        print(f"Image shape: {image.shape}")
        
        # Extract features using FeatureExtractor
        features = feature_extractor_test(image)
        feature_extractor_test.remove_handler()
        
        print(f"Features from 'decoders.2' shape: {features['decoders.2'][0].shape}")
        print(f"Features from 'decoders.2.basic_module.SingleConv2.ReLU' shape: {features['decoders.2.basic_module.SingleConv2.ReLU'][0].shape}")
        
        # Check if they are equal
        are_equal = torch.allclose(
            features['decoders.2'][0], 
            features['decoders.2.basic_module.SingleConv2.ReLU'][0], 
            atol=1e-5
        )
        print(f"Are features equal? {are_equal}")
        
        # Get model output with features
        pred, feats = model(image)
        print(f"Model decoder features shape: {feats[-1].shape}")
        
        # Check which one matches the model's decoder features
        matches_decoder2 = torch.allclose(features['decoders.2'][0], feats[-1].detach(), atol=1e-5)
        matches_relu = torch.allclose(features['decoders.2.basic_module.SingleConv2.ReLU'][0], feats[-1].detach(), atol=1e-5)
        
        print(f"decoders.2 matches model output: {matches_decoder2}")
        print(f"decoders.2.basic_module.SingleConv2.ReLU matches model output: {matches_relu}")
        
        break

Image shape: torch.Size([1, 1, 256, 256])
Features from 'decoders.2' shape: torch.Size([1, 32, 256, 256])
Features from 'decoders.2.basic_module.SingleConv2.ReLU' shape: torch.Size([1, 32, 256, 256])
Are features equal? False
Model decoder features shape: torch.Size([1, 32, 256, 256])
decoders.2 matches model output: False
decoders.2.basic_module.SingleConv2.ReLU matches model output: True


## Explanation of the Feature Extraction Issue

The problem is in the `FeatureExtractor` class implementation. The forward hook is capturing the **input** to layers, not the **output**:

```python
def save_outputs_hook(self, layer_id: str) -> Callable:
    def fn(_, input, output):
        self._features[layer_id] = input  # <-- Captures INPUT, not OUTPUT!
    return fn
```

This means:
- `'decoders.2'` captures the **input** to the decoder (before processing)
- `'decoders.2.basic_module.SingleConv2.ReLU'` captures the **input** to ReLU (which is the output after conv+batchnorm)
- The model's `feats[-1]` returns the actual **output** of the decoder

That's why `'decoders.2.basic_module.SingleConv2.ReLU'` matches the model output - it's getting the processed features right before the final ReLU activation.

In [36]:
# Let's investigate what feats[-1] actually contains by examining the model forward pass
print("=== INVESTIGATING feats[-1] CONTENTS ===")

# Let's trace through the decoder forward pass step by step
with torch.no_grad():
    for i, (image, label) in enumerate(datasets[0]):
        print(f"Image shape: {image.shape}")
        
        # Get the model's feature output
        pred, feats = model(image)
        print(f"Number of decoder features returned: {len(feats)}")
        print(f"feats[-1] shape: {feats[-1].shape}")
        
        # Let's also check what happens if we manually extract features after ReLU
        layers_after_relu = ['decoders.2']
        feature_extractor_after = FeatureExtractor(model, layers_after_relu)
        
        features_after = feature_extractor_after(image)
        feature_extractor_after.remove_handler()
        
        print(f"Features after complete decoders.2 processing: {features_after['decoders.2'][0].shape}")
        
        # Compare feats[-1] with the output after the complete decoder
        matches_after_decoder = torch.allclose(feats[-1], features_after['decoders.2'][0], atol=1e-5)
        print(f"feats[-1] matches output after complete decoders.2: {matches_after_decoder}")
        
        break

=== INVESTIGATING feats[-1] CONTENTS ===
Image shape: torch.Size([1, 1, 256, 256])
Number of decoder features returned: 4
feats[-1] shape: torch.Size([1, 32, 256, 256])
Features after complete decoders.2 processing: torch.Size([1, 32, 256, 256])
feats[-1] matches output after complete decoders.2: False


In [37]:
# Let's create a custom hook to capture BOTH input and output to see what's happening
print("=== DETAILED HOOK INVESTIGATION ===")

class DetailedFeatureExtractor(torch.nn.Module):
    def __init__(self, model, layers):
        super().__init__()
        self.model = model
        self.layers = layers
        self._inputs = {layer: None for layer in layers}
        self._outputs = {layer: None for layer in layers}
        self.handlers = []
        
        for layer_id in layers:
            layer = dict([*self.model.named_modules()])[layer_id]
            handler = layer.register_forward_hook(self.save_both_hook(layer_id))
            self.handlers.append(handler)
    
    def save_both_hook(self, layer_id: str):
        def fn(module, input, output):
            self._inputs[layer_id] = input[0] if isinstance(input, tuple) else input
            self._outputs[layer_id] = output
        return fn
    
    def forward(self, x):
        _ = self.model(x)
        return self._inputs, self._outputs
    
    def remove_handlers(self):
        for handler in self.handlers:
            handler.remove()

# Test with the ReLU layer
detailed_extractor = DetailedFeatureExtractor(model, ['decoders.2.basic_module.SingleConv2.ReLU'])

with torch.no_grad():
    for i, (image, label) in enumerate(datasets[0]):
        inputs, outputs = detailed_extractor(image)
        detailed_extractor.remove_handlers()
        
        # Get model features
        pred, feats = model(image)
        
        relu_input = inputs['decoders.2.basic_module.SingleConv2.ReLU']
        relu_output = outputs['decoders.2.basic_module.SingleConv2.ReLU']
        
        print(f"ReLU input shape: {relu_input.shape}")
        print(f"ReLU output shape: {relu_output.shape}")
        print(f"Model feats[-1] shape: {feats[-1].shape}")
        
        print(f"ReLU input matches feats[-1]: {torch.allclose(relu_input, feats[-1], atol=1e-5)}")
        print(f"ReLU output matches feats[-1]: {torch.allclose(relu_output, feats[-1], atol=1e-5)}")
        
        # Check if ReLU input and output are the same (which would indicate ReLU is not changing values)
        print(f"ReLU input equals output: {torch.allclose(relu_input, relu_output, atol=1e-5)}")
        print(f"Min values - Input: {relu_input.min():.6f}, Output: {relu_output.min():.6f}")
        
        break

=== DETAILED HOOK INVESTIGATION ===
ReLU input shape: torch.Size([1, 32, 256, 256])
ReLU output shape: torch.Size([1, 32, 256, 256])
Model feats[-1] shape: torch.Size([1, 32, 256, 256])
ReLU input matches feats[-1]: True
ReLU output matches feats[-1]: True
ReLU input equals output: True
Min values - Input: 0.000000, Output: 0.000000


In [25]:
# Let's investigate the batch dimension issue
print("=== BATCH DIMENSION INVESTIGATION ===")

with torch.no_grad():
    for i, (image, label) in enumerate(dataloaders[0]):
        print(f"Original batch shape: {image.shape}")
        print(f"Label batch shape: {label.shape}")
        
        # Remove the extra dimension if present
        if image.dim() == 5:
            image = torch.squeeze(image, dim=-3)
        
        print(f"After squeeze - Image shape: {image.shape}")
        
        # Test the FeatureExtractor with batch
        layers_test = ['decoders.2']
        feature_extractor_batch = FeatureExtractor(model, layers_test)
        
        features = feature_extractor_batch(image)
        feature_extractor_batch.remove_handler()
        
        print(f"FeatureExtractor output shape: {features[layers_test[0]].shape}")
        print(f"Expected shape should be: [{image.shape[0]}, 32, {image.shape[2]}, {image.shape[3]}]")
        
        # Let's also check what the model returns
        pred, feats = model(image)
        print(f"Model prediction shape: {pred.shape}")
        print(f"Model feats[-1] shape: {feats[-1].shape}")
        
        # Check if the issue is in how we're indexing the features
        print(f"Type of features[layers_test[0]]: {type(features[layers_test[0]])}")
        print(f"Features keys: {features.keys()}")
        
        break

=== BATCH DIMENSION INVESTIGATION ===
Original batch shape: torch.Size([10, 1, 1, 256, 256])
Label batch shape: torch.Size([10, 1, 1, 256, 256])
After squeeze - Image shape: torch.Size([10, 1, 256, 256])
FeatureExtractor output shape: torch.Size([10, 32, 256, 256])
Expected shape should be: [10, 32, 256, 256]
Model prediction shape: torch.Size([10, 1, 256, 256])
Model feats[-1] shape: torch.Size([10, 32, 256, 256])
Type of features[layers_test[0]]: <class 'torch.Tensor'>
Features keys: dict_keys(['decoders.2'])


In [26]:
# Let's check the difference between accessing features directly vs with [0] indexing
print("=== INDEXING INVESTIGATION ===")

with torch.no_grad():
    for i, (image, label) in enumerate(dataloaders[0]):
        if image.dim() == 5:
            image = torch.squeeze(image, dim=-3)
        
        print(f"Batch size: {image.shape[0]}")
        
        layers_test = ['decoders.2']
        feature_extractor_test = FeatureExtractor(model, layers_test)
        
        features = feature_extractor_test(image)
        feature_extractor_test.remove_handler()
        
        # Check different ways of accessing the features
        print(f"features[layers_test[0]].shape: {features[layers_test[0]].shape}")
        print(f"features[layers_test[0]][0].shape: {features[layers_test[0]][0].shape}")  # This is taking the first item in batch!
        
        # Show the issue
        print("\n=== THE ISSUE ===")
        print("When you use features[layers[0]][0], you're taking the FIRST sample from the batch!")
        print("- features[layers[0]] has shape [batch_size, channels, height, width]")
        print("- features[layers[0]][0] has shape [channels, height, width] - just the first sample!")
        
        break

=== INDEXING INVESTIGATION ===
Batch size: 10
features[layers_test[0]].shape: torch.Size([10, 32, 256, 256])
features[layers_test[0]][0].shape: torch.Size([32, 256, 256])

=== THE ISSUE ===
When you use features[layers[0]][0], you're taking the FIRST sample from the batch!
- features[layers[0]] has shape [batch_size, channels, height, width]
- features[layers[0]][0] has shape [channels, height, width] - just the first sample!


In [36]:
# Let's demonstrate the hook removal issue
print("=== HOOK REMOVAL ISSUE DEMONSTRATION ===")

# First, let's show the WRONG way (removing handlers inside the loop)
print("WRONG WAY - Removing handlers inside loop:")
layers_test = ['decoders.2.basic_module.SingleConv2.ReLU']
feature_extractor_wrong = FeatureExtractor(model, layers_test)

with torch.no_grad():
    for i, (image, label) in enumerate(datasets[0]):
        print(f"\nIteration {i}:")
        
        features = feature_extractor_wrong(image)
        feature_extractor_wrong.remove_handler()  # ❌ WRONG: Removing handlers here!
        
        pred, feats = model(image)
        
        features_equal = torch.allclose(features[layers_test[0]], feats[-1].detach(), atol=1e-5)
        print(f"Features equal: {features_equal}")
        
        if i > 2:  # Just test a few iterations
            break

print("\n" + "="*50)

# Now let's show the CORRECT way (removing handlers after the loop)
print("CORRECT WAY - Removing handlers after loop:")
feature_extractor_correct = FeatureExtractor(model, layers_test)

with torch.no_grad():
    for i, (image, label) in enumerate(datasets[0]):
        print(f"\nIteration {i}:")
        
        features = feature_extractor_correct(image)
        # ✅ CORRECT: Don't remove handlers here!
        
        pred, feats = model(image)
        
        features_equal = torch.allclose(features[layers_test[0]], feats[-1].detach(), atol=1e-5)
        print(f"Features equal: {features_equal}")
        
        if i > 2:  # Just test a few iterations
            break
    
    # ✅ CORRECT: Remove handlers after the loop is finished
    feature_extractor_correct.remove_handler()

=== HOOK REMOVAL ISSUE DEMONSTRATION ===
WRONG WAY - Removing handlers inside loop:

Iteration 0:
Features equal: True

Iteration 1:
Features equal: False

Iteration 2:
Features equal: False

Iteration 3:
Features equal: False

CORRECT WAY - Removing handlers after loop:

Iteration 0:
Features equal: True

Iteration 1:
Features equal: True

Iteration 2:
Features equal: True

Iteration 3:
Features equal: True


## Explanation of the Hook Removal Issue

**The Problem:**
When you call `feature_extractor.remove_handler()` inside the loop, you're removing the forward hooks from the PyTorch model. Once these hooks are removed, they don't get automatically re-registered, so subsequent calls to `feature_extractor(image)` can't capture features anymore.

**What happens step by step:**

1. **First iteration (i=0):**
   - Hooks are active ✅
   - `feature_extractor(image)` captures features correctly
   - `remove_handler()` removes all hooks ❌
   - Features match model output ✅

2. **Second iteration (i=1):**
   - Hooks are gone ❌
   - `feature_extractor(image)` can't capture anything (returns stale/empty tensors)
   - Features don't match model output ❌

**The Solution:**
- Either remove handlers **after** the entire loop finishes
- Or recreate the `FeatureExtractor` inside each iteration (less efficient)
- Or don't remove handlers until you're completely done with feature extraction

**Best Practice:**
```python
feature_extractor = FeatureExtractor(model, layers)
try:
    # Your loop here
    pass
finally:
    feature_extractor.remove_handler()  # Always cleanup
```